Parameters Module
=================

The ``brisket.parameters`` module is responsible for handling the parameters used in ``brisket``. 
This module provides classes and methods to manage, validate, and manipulate parameters for different models such as galaxies and AGN. 

Classes
-------

- ``Params``: This is the main class for handling parameters. It allows adding sources, validating parameters, and provides a summary of fixed and free parameters.
- ``Group``: A class representing a parameter group, used for further sub-dividing the parameter specification. The ``Group`` class serves as a container for parameters belonging to a given source (e.g. galaxy, AGN) or absorber (e.g. dust) and can have its own sub-Groups.
- ``FreeParam``: A class representing a free parameter with specified limits and prior distributions.
- ``FixedParam``: A class representing a fixed parameter with a constant value. In practice, this is generally not used, as fixed parameters can be provided as integers or floats directly.

Usage
-----

You can initialize a ``Params`` object with a template (see: :doc:`templates`.) or as an emtpy object:

In [1]:
import brisket
params = brisket.Params()

From there, you can then add sources and parameters as needed.
There is a huge diversity of potential model specifications, but all models require the redshift to be specified at the base level in the ``Params`` object. 
Here, we'll allow the redshift to be free from $z\sim 6$-$8$, and apply a Gaussian prior centered at $z=7$. 

In [2]:
params['redshift'] = brisket.FreeParam(6, 8, prior='Normal', mu=7, sigma=0.2)

Note that we could have specified ``prior='Gaussian'`` and it would work the same way; multiple aliases exist for a given prior. 


From here, the parameter structure of ``brisket`` is broken up into sources (sources of emission), absorbers (things that absorb emission), and reprocessors (things that absorb emission and re-emit as sources of emission). 
There are also calibrators, but we'll get into that later. 
As a simple example, we can add a source called ``'galaxy'``, noting that the name ``'galaxy'`` is special and calls the default stellar model ``brisket.models.StellarModel``. 

In [3]:
class NullModel:
    order = 100
    def __init__(self, params):
        self.params = params
        
params.add_source('galaxy', model=NullModel)
params['galaxy']['logMstar'] = brisket.FreeParam(6, 12)
params['galaxy']['zmet'] = brisket.FreeParam(0.001, 1, prior='log_uniform')

Printing the parameters object will provide a nice representation of the parameter structure, including the fixed and free parameters:

In [4]:
print(params)


                       Free Parameters                        
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter name  ┃   Limits   ┃ Prior                       ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ redshift        │   (6, 8)   │ Norm(6, 8, mu=7, sigma=0.2) │
│ galaxy/logMstar │  (6, 12)   │ Uniform(6, 12)              │
│ galaxy/zmet     │ (0.001, 1) │ LogUniform(0.001, 1)        │
└─────────────────┴────────────┴─────────────────────────────┘



Defaults and Aliases
--------------------

We include several aliases for adding sources/absorbers/reprocessors to the params object. For example, 

In [5]:
params.add_igm()

is an alias for the slightly longer expression

In [ ]:

params.add_absorber('igm', model=briskest.models.InoueIGMModel)


You'll notice that the ``add_igm()`` methods presumes the default ``InoueIGMModel`` model, though this can be changed by passing a different model, e.g. ``params.add_igm(model=MadauIGMModel)``.
In any case, the ``add_absorber()`` method is iself an alias for the multi-step process of initializing a "parameter group" to describe the IGM model, noting that it is an "absorber," and adding it to the params object: 

In [ ]:
igm = brisket.parameters.Group('igm', model=briskest.models.InoueIGMModel, model_type='absorber')
params['igm'] = igm


This is a bit more verbose, but allows for more flexibility in the parameter structure, and allows you to specify your own custom models. Say, for example, you wanted to include in your model a Damped Lyman-alpha system, you could define a custom DLA absorbption class and add it to the params object like so:

In [ ]:
class CustomDLAModel(brisket.models.BaseIGMModel):
  type = 'absorber' # tells the code how to treat this model (technically, not necessary since its inherited from BaseIGMModel)
  order = 100 # tells the code how to treat this model (technically, not necessary since its inherited from BaseIGMModel)
  def __init__(self, **kwargs):
    super().__init__(**kwargs)

  def absorb(self, sed_incident):
    # custom absorption code here
    return sed_absorbed

dla = brisket.parametrs.Group('dla', model=CustomDLAModel, model_type='absorber')
params['dla'] = dla


More details are provided in the :doc:`custom_models` documentation.




<!-- .. Implemented by default: 

.. - Galaxy (Source)
..     - SFH (Group)
.. - AGN (Source)
.. - Nebular (Reprocessor)
.. - Dust (Reprocessor)
.. - IGM (Absorber)
.. - Calibration (Group) -->